In [1]:
# Cell 1 — imports
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from config import CONFIG

engine = create_engine(CONFIG["db_url"])
print("Connected")

Connected


In [15]:
# Cell 2 — load raw table
# Always load from raw source at the start of the cleaning notebook
df = pd.read_sql(
    f'SELECT * FROM "{CONFIG["schema"]}"."{CONFIG["raw_table"]}"',
    engine
)
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")

Loaded: 101,766 rows x 50 columns


In [17]:
#  define sentinel replacement function
#replacing sentinals with NAN

def replace_sentinel(df, sentinel):
    """
    Replace sentinel string values with numpy NaN.
    Operates column by column on string columns only
    to handle pandas 4 mixed-type object columns correctly.
    """
    df_out      = df.copy()
    total_count = 0
    string_cols = df_out.select_dtypes(include=["object", "str"]).columns

    for col in string_cols:
        col_count = (df_out[col] == sentinel).sum()
        if col_count > 0:
            df_out[col]  = df_out[col].replace(sentinel, np.nan)
            total_count += col_count

    print(f"Replaced {total_count:,} '{sentinel}' values with NaN")
    return df_out


In [18]:
df = replace_sentinel(df, CONFIG["missing_sentinel"])


Replaced 192,849 '?' values with NaN


In [5]:
# Verify: null counts should now match sentinel counts from profiling
print("\nNull counts after sentinel replacement:")
new_nulls = df.isna().sum()
print(new_nulls[new_nulls > 0].sort_values(ascending=False).to_string())



Null counts after sentinel replacement:
weight               98569
medical_specialty    49949
payer_code           40256
race                  2273
diag_3                1423
diag_2                 358
diag_1                  21
admission_type_id        1


## Sentinel Value Replacement — Findings

**Purpose:**
Replace all "?" sentinel strings with numpy NaN so pandas correctly
identifies missing values, excludes them from aggregations, and produces
accurate null counts in all downstream cleaning steps.

**Result: 192,850 sentinel values replaced across 8 columns**

| Column | Null Count | Null % | Treatment Decision |
|---|---|---|---|
| weight | 98,569 | 96.86% | Add weight_available flag — too sparse to use |
| medical_specialty | 49,949 | 49.08% | Fill "Unknown" — imputation would be guessing |
| payer_code | 40,256 | 39.56% | Fill "Unknown" — not clinically interpretable |
| race | 2,273 | 2.23% | Fill "Unknown" — demographics never imputed |
| diag_3 | 1,423 | 1.40% | Fill "None" — tertiary diagnosis legitimately absent |
| diag_2 | 358 | 0.35% | Fill "None" — secondary diagnosis legitimately absent |
| diag_1 | 21 | 0.02% | Flag as diag_1_missing_flag — primary diagnosis missing is a data quality error |
| admission_type_id | 1 | 0.00% | Genuine NULL from raw data — not a sentinel value |

**Key Observations:**

1. **weight is 96.86% missing** — only 3,197 out of 101,766 rows have
   a recorded weight value. No imputation strategy is appropriate at
   this level of missingness. A binary flag (weight_available) will
   be created to indicate whether weight was recorded, preserving
   the information that weight data exists for a small subset of patients.

2. **medical_specialty and payer_code are high-missing** — together
   accounting for ~89,000 missing values. Both will be filled with
   "Unknown" rather than dropped. Dropping rows would reduce the
   dataset by nearly half and introduce selection bias into all
   readmission rate calculations.

3. **race is low-missing at 2.23%** — 2,273 rows. Filled with
   "Unknown". Demographics must never be imputed — assigning a
   race to a patient based on statistical patterns would be both
   clinically and ethically inappropriate.

4. **diag_2 and diag_3 are legitimately absent** — a patient with
   a straightforward single diagnosis will have no secondary or
   tertiary diagnosis codes. These are not data quality errors.
   Filled with "None" to distinguish from missing vs not applicable.

5. **diag_1 has only 21 missing values** — a missing primary diagnosis
   is a genuine data quality issue. These 21 rows will be flagged
   with diag_1_missing_flag = 1 rather than dropped, preserving
   them for audit purposes while excluding them from diagnosis
   category analysis.

6. **admission_type_id has 1 null** — this was a genuine NULL in the
   raw data confirmed by the structural profile in Step 2. It was
   not a sentinel value. It will be handled in the dtype correction
   step when admission_type_id is cast to string.

**Verification:**
Null counts now correctly reflect the true missingness in the dataset.
The structural profile in Step 2 showed zero nulls because all missing
values were hidden as "?" strings. The counts above represent the
actual missing data picture that will guide all treatment decisions
in the steps that follow.

**Next Step:** Duplicate detection and removal.

In [6]:
# deduplication.py

def remove_duplicates(df, pk, patient_key):
    original_len = len(df)

    # Step 1: Remove exact full-row duplicates
    df_out       = df.drop_duplicates()
    full_removed = original_len - len(df_out)
    print(f"Full row duplicates removed  : {full_removed:,}")

    # Step 2: Detect primary key conflicts (different data, same PK)
    pk_conflict_mask = df_out.duplicated(subset=[pk], keep=False)
    conflict_count   = pk_conflict_mask.sum()

    if conflict_count > 0:
        print(f"Primary key conflicts found  : {conflict_count:,}  flagged")
        df_out["duplicate_flag"] = pk_conflict_mask.astype(int)
    else:
        print("No primary key conflicts found.")
        df_out["duplicate_flag"] = 0

    # Encounter frequency per patient — expected to have repeats
    enc_per_patient = df_out.groupby(patient_key)[pk].count()
    print(f"\nPatients with >1 encounter   : {(enc_per_patient > 1).sum():,}")
    print(f"Max encounters per patient   : {enc_per_patient.max()}")

    return df_out

df = remove_duplicates(df, CONFIG["primary_key"], CONFIG["patient_key"])

Full row duplicates removed  : 0
No primary key conflicts found.

Patients with >1 encounter   : 16,773
Max encounters per patient   : 40


In [7]:
# missing_treatment.py

def treat_missing(df):
    df_out = df.copy()

    # Weight: too sparse to impute — flag its presence instead
    df_out["weight_available"] = df_out["weight"].notna().astype(int)
    print(f"weight_available flag  | Present in {df_out['weight_available'].sum():,} rows")

    # High-missing categoricals: fill "Unknown" — never drop rows for these
    for col in ["payer_code", "medical_specialty", "race"]:
        n = df_out[col].isna().sum()
        df_out[col] = df_out[col].fillna("Unknown")
        print(f"{col:<22}: {n:,} nulls filled with Unknown")

    # Secondary/tertiary diagnoses: legitimately absent — fill "None"
    for col in ["diag_2", "diag_3"]:
        n = df_out[col].isna().sum()
        df_out[col] = df_out[col].fillna("None")
        print(f"{col:<22}: {n:,} nulls filled with None")

    # Primary diagnosis: missing is a data quality error — flag
    n_d1 = df_out["diag_1"].isna().sum()
    df_out["diag_1_missing_flag"] = df_out["diag_1"].isna().astype(int)
    if n_d1 > 0:
        print(f"\nWARNING: {n_d1:,} rows missing primary diagnosis (diag_1) — flagged")

    # Final check
    remaining = df_out.isna().sum()
    remaining = remaining[remaining > 0]
    if len(remaining) == 0:
        print("\nNo missing values remain.")
    else:
        print(f"\nRemaining nulls:\n{remaining.to_string()}")

    return df_out

df = treat_missing(df)

weight_available flag  | Present in 3,197 rows
payer_code            : 40,256 nulls filled with Unknown
medical_specialty     : 49,949 nulls filled with Unknown
race                  : 2,273 nulls filled with Unknown
diag_2                : 358 nulls filled with None
diag_3                : 1,423 nulls filled with None


Remaining nulls:
weight               98569
admission_type_id        1
diag_1                  21


## Missing Value Treatment — Findings

**Purpose:**
Apply targeted treatment to each column based on the missingness
percentage and clinical meaning established during profiling. The
rule throughout is flag or fill — never silently drop rows.

**Treatment Applied:**

| Column | Nulls | Treatment | Clinical Reason |
|---|---|---|---|
| weight | 98,569 | weight_available flag added | 96.86% missing — too sparse to impute. Flag preserves the information that weight exists for 3,197 patients |
| medical_specialty | 49,949 | Filled with "Unknown" | ~49% missing — imputing a clinical specialty would fabricate analytical categories |
| payer_code | 40,256 | Filled with "Unknown" | ~40% missing — administrative field, not clinically interpretable |
| race | 2,273 | Filled with "Unknown" | ~2% missing — demographics are never imputed under any circumstances |
| diag_3 | 1,423 | Filled with "None" | Tertiary diagnosis legitimately absent for simpler cases |
| diag_2 | 358 | Filled with "None" | Secondary diagnosis legitimately absent for simpler cases |
| diag_1 | 21 | diag_1_missing_flag = 1 added | Primary diagnosis missing is a data quality error — flagged not dropped |

**Key Observations:**

1. **weight_available flag created** — only 3,197 out of 101,766 rows
   have a recorded weight value (3.14% of encounters). The flag allows
   the small subset of patients with weight data to be analyzed
   separately if needed, without fabricating weight values for the
   other 96.86%.

2. **"Unknown" vs "None" distinction is intentional** — "Unknown"
   is used for columns where data was collected but not recorded
   (payer_code, medical_specialty, race). "None" is used for
   diagnosis columns where the absence of a value means the condition
   genuinely does not apply. These are clinically different meanings
   and must be treated differently in SQL filtering.

3. **21 diag_1 rows flagged** — a missing primary diagnosis is not
   a legitimate absence like diag_2 or diag_3. Every inpatient
   encounter should have a primary diagnosis code. These 21 rows
   represent a data entry or extraction error and are flagged with
   diag_1_missing_flag = 1 so they can be excluded from diagnosis
   category analysis in the SQL EDA phase without being silently lost.

4. **Three columns still show remaining nulls** — this is expected
   and intentional:

   - **weight (98,569)** — retained as NaN deliberately. The column
     is kept for audit trail purposes. The weight_available flag
     carries the analytical information forward.

   - **admission_type_id (1)** — a single genuine NULL that was in
     the raw data before sentinel replacement. It will become the
     string "nan" when cast to string in the dtype correction step,
     which is acceptable for a single row in a lookup code column.

   - **diag_1 (21)** — retained as NaN deliberately. These rows are
     flagged with diag_1_missing_flag = 1. They remain in the dataset
     for row count integrity but will be excluded from diagnosis
     category analysis by filtering on the flag column.

**Total nulls treated: 134,259 values across 6 columns**

134,259 were filled with Unknown, None, or flagged
98,591 remain as intentional NaN

The 21 diag_1 nulls and the single admission_type_id null were genuine NULLs from the raw data, not sentinels — which is why the numbers do not add up to exactly 192,850

In [8]:
# dtype_correction.py

def correct_dtypes(df, id_cols, numeric_cols):
    df_out = df.copy()

    # ID columns: cast to string, strip the ".0" pandas adds when int passes through float
    for col in id_cols:
        df_out[col] = df_out[col].astype(str).str.split(".").str[0]
    print(f"Cast to string (categorical ID cols): {id_cols}")

    # Numeric columns: coerce and report any new nulls created
    for col in numeric_cols:
        before = df_out[col].dtype
        df_out[col] = pd.to_numeric(df_out[col], errors="coerce")
        new_nulls   = df_out[col].isna().sum()
        print(f"{col:<25}: {before} -> {df_out[col].dtype} | coercion nulls: {new_nulls}")

    return df_out

df = correct_dtypes(df, CONFIG["id_cols"], CONFIG["numeric_cols"])

Cast to string (categorical ID cols): ['admission_type_id', 'discharge_disposition_id', 'admission_source_id']
time_in_hospital         : int64 -> int64 | coercion nulls: 0
num_lab_procedures       : int64 -> int64 | coercion nulls: 0
num_procedures           : int64 -> int64 | coercion nulls: 0
num_medications          : int64 -> int64 | coercion nulls: 0
number_outpatient        : int64 -> int64 | coercion nulls: 0
number_emergency         : int64 -> int64 | coercion nulls: 0
number_inpatient         : int64 -> int64 | coercion nulls: 0
number_diagnoses         : int64 -> int64 | coercion nulls: 0


## Step 9: Data Type Correction — Findings

**Purpose:**
Correct data types for two groups of columns:
1. ID lookup columns stored as integers in the source — cast to string
   because they are categorical codes, not quantities
2. Numeric columns — verified as correct integer types with no
   coercion errors

**Results:**

**ID Columns Cast to String:**

| Column | Before | After | Reason |
|---|---|---|---|
| admission_type_id | int64/float64 | string | Lookup code — e.g. 1=Emergency, 3=Elective. Summing or averaging these codes is meaningless |
| discharge_disposition_id | int64 | string | Lookup code — e.g. 1=Home, 3=SNF. Arithmetic on these values would produce nonsensical results |
| admission_source_id | int64 | string | Lookup code — e.g. 7=Emergency Room, 1=Physician Referral |

**Numeric Columns Verified:**

| Column | Before | After | Coercion Nulls |
|---|---|---|---|
| time_in_hospital | int64 | int64 | 0 |
| num_lab_procedures | int64 | int64 | 0 |
| num_procedures | int64 | int64 | 0 |
| num_medications | int64 | int64 | 0 |
| number_outpatient | int64 | int64 | 0 |
| number_emergency | int64 | int64 | 0 |
| number_inpatient | int64 | int64 | 0 |
| number_diagnoses | int64 | int64 | 0 |

**Key Observations:**

1. **All three ID columns successfully cast to string** — these columns
   arrived as integers because the source database stored the lookup
   codes numerically. Casting to string prevents any downstream tool
   — Python, SQL, or Power BI — from accidentally treating them as
   measurable quantities. For example, averaging discharge_disposition_id
   values would produce a number like 4.7 which has no clinical meaning.

2. **The .0 suffix was stripped** — admission_type_id arrived as
   float64 due to its single NULL value (pandas promotes integer
   columns to float when NULLs are present). The cast to string
   would have produced values like "6.0" instead of "6". The
   str.split(".").str[0] step strips the decimal suffix so values
   read as "6", "1", "3" — matching the IDs_mapping.csv lookup
   table correctly.

3. **All 8 numeric columns confirmed as int64** — no dtype changes
   were needed and no coercion nulls were introduced. This confirms
   the numeric columns arrived from PostgreSQL in the correct format
   and contain no hidden string values or unexpected characters that
   would cause pd.to_numeric() to fail.

4. **Zero coercion nulls across all numeric columns** — if any numeric
   column had contained a non-numeric string value, pd.to_numeric()
   with errors="coerce" would have converted it to NaN and the
   coercion null count would be greater than zero. All zeros here
   confirms the numeric data is clean and consistent.

**Why This Step Matters for Downstream Tools:**

- **PostgreSQL:** String ID columns will GROUP BY correctly as
  categorical labels rather than being sorted numerically
- **Power BI:** Will not offer SUM or AVERAGE options on ID columns
  since they are recognized as text fields
- **SQL KPI views:** The CASE statements mapping disposition codes
  to labels (e.g. WHEN '1' THEN 'Discharged to Home') require
  string comparison — if the column were still integer the WHEN
  conditions would never match

**Next Step:** Text standardization — strip whitespace, standardize
casing across all string columns, and rename hyphenated medication
column names to snake_case for SQL compatibility.

In [9]:
# text_standardization.py
import pandas as pd

def standardize_text(df):
    df_out = df.copy()

    # Strip + title case all object columns
    for col in df_out.select_dtypes(include=["object", "string"]).columns:
        df_out[col] = df_out[col].str.strip().str.title()

    # readmitted uses uppercase values — override title case
    df_out["readmitted"] = df_out["readmitted"].str.upper()

    # Verify controlled vocabulary columns post-standardization
    for col in ["gender", "readmitted", "change", "diabetesMed"]:
        print(f"{col:<15}: {sorted(df_out[col].dropna().unique())}")

    # Rename hyphenated medication columns to snake_case
    rename_map = {
        "glyburide-metformin"      : "glyburide_metformin",
        "glipizide-metformin"      : "glipizide_metformin",
        "glimepiride-pioglitazone" : "glimepiride_pioglitazone",
        "metformin-rosiglitazone"  : "metformin_rosiglitazone",
        "metformin-pioglitazone"   : "metformin_pioglitazone"
    }
    df_out = df_out.rename(columns=rename_map)
    print(f"\nRenamed: {list(rename_map.values())}")
    return df_out

In [10]:
df = standardize_text(df)

gender         : ['Female', 'Male', 'Unknown/Invalid']
readmitted     : ['<30', '>30', 'NO']
change         : ['Ch', 'No']
diabetesMed    : ['No', 'Yes']

Renamed: ['glyburide_metformin', 'glipizide_metformin', 'glimepiride_pioglitazone', 'metformin_rosiglitazone', 'metformin_pioglitazone']


## Text Standardization — Findings

**Purpose:**
Strip whitespace, standardize casing across all string columns, and
rename hyphenated medication column names to snake_case. Inconsistent
casing creates phantom groups in SQL GROUP BY queries and Power BI
visuals — for example "male", "Male", and "MALE" would be counted
as three separate categories instead of one.

**Results:**

**Controlled Vocabulary Verification:**

| Column | Unique Values Found | Status |
|---|---|---|
| gender | Female, Male, Unknown/Invalid | Expected — 3 values confirmed |
| readmitted | <30, >30, NO | Expected — 3 target classes confirmed |
| change | Ch, No | Expected — 2 values confirmed |
| diabetesMed | No, Yes | Expected — 2 values confirmed |

**Medication Columns Renamed to Snake Case:**

| Original Name | Renamed To |
|---|---|
| glyburide-metformin | glyburide_metformin |
| glipizide-metformin | glipizide_metformin |
| glimepiride-pioglitazone | glimepiride_pioglitazone |
| metformin-rosiglitazone | metformin_rosiglitazone |
| metformin-pioglitazone | metformin_pioglitazone |

**Key Observations:**

1. **All controlled vocabulary columns contain only expected values**
   — gender, readmitted, change, and diabetesMed all verify cleanly
   with no unexpected entries beyond what was identified during
   profiling. This confirms that title case standardization did not
   introduce any new unexpected values.

2. **readmitted correctly shows uppercase values** — the target
   column uses uppercase NO, <30, and >30 as designed. The
   str.upper() override applied after title case standardization
   correctly preserved these values. If title case had been applied
   without the override, "NO" would have become "No" which would
   have conflicted with the "No" value in change and diabetesMed,
   creating ambiguity.

3. **gender still shows Unknown/Invalid** — this value survived
   standardization correctly. It will be flagged as
   gender_invalid_flag = 1 in the next step and excluded from all
   readmission rate calculations. It is not cleaned away here
   because flagging preserves the audit trail.

4. **Five hyphenated medication columns renamed** — hyphens are
   invalid characters in SQL column identifiers. Without renaming,
   every SQL query referencing these columns would require quoted
   identifiers ("glyburide-metformin") which is error-prone and
   inconsistent. Snake_case names (glyburide_metformin) work
   natively in PostgreSQL, Python, and Power BI without any
   special handling.

5. **Whitespace stripping applied across all string columns** —
   leading and trailing spaces are invisible in data previews but
   cause mismatches in comparisons and GROUP BY queries. For
   example "Caucasian " and "Caucasian" would be counted as two
   different race categories without stripping. This step ensures
   all string values are clean and consistent.

**Impact on Downstream Tools:**

- **PostgreSQL GROUP BY:** All string columns will now group
  correctly without phantom duplicate categories from casing
  or whitespace differences
- **Power BI filters and slicers:** Values will display
  consistently — "Female" will always be "Female" not a mix
  of "female", "FEMALE", and "Female"
- **SQL CASE statements:** String comparisons like
  WHEN gender = 'Female' will match correctly across all rows

**Next Step:** Invalid value flagging — flag gender Unknown/Invalid
rows and discharge disposition codes representing expired or
hospice patients for exclusion from readmission rate calculations.

In [11]:
# invalid_values.

def flag_invalid_gender(df, valid_genders):
    mask = (~df["gender"].isin(valid_genders)) & df["gender"].notna()
    df["gender_invalid_flag"] = mask.astype(int)
    print(f"Invalid gender flagged: {mask.sum():,}")
    if mask.any():
        print(df.loc[mask, "gender"].value_counts().to_string())
    return df

def flag_expired_encounters(df, expired_codes):
    """
    Patients who died or were transferred to hospice cannot be readmitted.
    Flag so SQL views can exclude them from readmission rate denominators.
    """
    disp_int = pd.to_numeric(df["discharge_disposition_id"], errors="coerce")
    mask     = disp_int.isin(expired_codes)
    df["expired_or_hospice_flag"] = mask.astype(int)
    print(f"Expired/hospice encounters flagged: {mask.sum():,}")
    return df

def validate_medication_values(df, med_cols, valid_values):
    """Check for unexpected values in medication columns."""
    updated_cols = [c.replace("-", "_") for c in med_cols]
    all_valid    = True
    for col in updated_cols:
        if col not in df.columns:
            continue
        bad = df[col][~df[col].isin(valid_values) & df[col].notna()].unique()
        if len(bad) > 0:
            print(f"Unexpected values in {col}: {bad}")
            all_valid = False
    if all_valid:
        print("All medication columns contain expected values only.")

df = flag_invalid_gender(df, CONFIG["valid_gender"])
df = flag_expired_encounters(df, CONFIG["expired_dispositions"])
validate_medication_values(df, CONFIG["medication_cols"], CONFIG["valid_med_values"])

Invalid gender flagged: 3
gender
Unknown/Invalid    3
Expired/hospice encounters flagged: 2,423
All medication columns contain expected values only.


##  Invalid Value Flagging — Findings

**Purpose:**
Flag values that exist (not NULL) but are clinically or logically
invalid. Two categories require flagging:
1. gender = "Unknown/Invalid" — not a usable demographic category
2. Discharge disposition codes representing death or hospice transfer
   — these patients cannot be readmitted and must be excluded from
   all readmission rate calculations

Flagging rather than dropping preserves the audit trail and keeps
row counts intact for reporting purposes.

**Results:**

| Flag Column | Rows Flagged | Pct of Dataset | Reason |
|---|---|---|---|
| gender_invalid_flag | 3 | 0.00% | gender = Unknown/Invalid |
| expired_or_hospice_flag | 2,423 | 2.38% | Discharge disposition codes 11,13,14,19,20,21 |

**Key Observations:**

1. **Only 3 rows have invalid gender** — a negligible proportion
   of the dataset (0.003%). These 3 rows will be excluded from
   all readmission rate calculations by filtering
   gender_invalid_flag = 0 in the SQL base view. Their exclusion
   has no meaningful impact on any KPI.

2. **2,423 encounters flagged as expired or hospice** — this is
   the most clinically critical flag in the entire cleaning phase.
   These patients were discharged to one of these dispositions:
   - Code 11: Expired (died in hospital)
   - Code 13: Hospice / Medical Facility
   - Code 14: Hospice / Home
   - Code 19: Expired at Home (Medicaid only)
   - Code 20: Expired in a Medical Facility (Medicaid only)
   - Code 21: Expired — Place Unknown (Medicaid only)

   A patient who died or entered hospice cannot be readmitted.
   Including these 2,423 encounters in the readmission rate
   denominator would artificially deflate the 30-day readmission
   rate by inflating the NOT READMITTED count.

3. **Impact on readmission rate if not excluded:**
   - Total encounters including expired/hospice : 101,766
   - Total encounters excluding expired/hospice : 99,343
   - <30 readmissions                           : 11,357
   - Rate INCLUDING expired (incorrect)         : 11.16%
   - Rate EXCLUDING expired (correct)           : 11,357 / 99,343 = 11.43%

   The difference of 0.27 percentage points may appear small but
   represents a meaningful clinical distinction — a hospital
   reporting 11.16% when the correct rate is 11.43% is
   understating its readmission burden. This is exactly the
   kind of methodological decision that CMS and global health
   organizations scrutinize when evaluating hospital performance.

4. **All medication columns contain expected values only** —
   all 23 medication columns contain only the four expected
   values: No, Steady, Up, Down. No unexpected entries,
   free-text values, or encoding errors exist in any medication
   column. This confirms the medication data arrived consistently
   structured and requires no additional cleaning beyond what
   has already been applied.

5. **Both flags are binary columns (0 or 1)** — they will travel
   with the data into diabetic_data_clean in PostgreSQL and are
   applied as WHERE filters in the SQL base view vw_encounter_base:
   WHERE expired_or_hospice_flag = 0
     AND gender_invalid_flag     = 0
   This single filter in one place ensures all 8 KPI views
   automatically exclude invalid records without repeating
   the logic in every view.

**Corrected Valid Analysis Population:**
- Starting rows                    : 101,766
- Expired or hospice (flagged)     : 2,423
- Invalid gender (flagged)         : 3
- Valid encounters for analysis    : 99,340
- This is the denominator for all readmission rate KPIs

**Next Step:** Outlier detection and flagging — identify statistical
outliers in all numeric columns using the IQR method and add
per-column flag columns before export.

In [12]:
# outlier_detection.py

def flag_iqr_outliers(df, numeric_cols, multiplier=1.5):
    """
    Adds one flag column per numeric column and a composite any_outlier_flag.
    multiplier=1.5  standard outliers
    multiplier=3.0  extreme outliers only
    """
    df_out  = df.copy()
    summary = []

    for col in numeric_cols:
        Q1  = df[col].quantile(0.25)
        Q3  = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lo  = Q1 - multiplier * IQR
        hi  = Q3 + multiplier * IQR

        mask             = (df[col] < lo) | (df[col] > hi)
        flag_col         = f"{col}_outlier_flag"
        df_out[flag_col] = mask.astype(int)

        summary.append({
            "column"        : col,
            "lower_bound"   : round(lo, 2),
            "upper_bound"   : round(hi, 2),
            "outlier_count" : int(mask.sum()),
            "outlier_pct"   : round(mask.mean() * 100, 2)
        })

    summary_df = pd.DataFrame(summary)
    print(summary_df.to_string(index=False))

    flag_cols = [f"{c}_outlier_flag" for c in numeric_cols]
    df_out["any_outlier_flag"] = (df_out[flag_cols].sum(axis=1) > 0).astype(int)
    print(f"\nRows with at least one outlier: "
          f"{df_out['any_outlier_flag'].sum():,} "
          f"({df_out['any_outlier_flag'].mean()*100:.1f}%)")

    summary_df.to_csv("outlier_report.csv", index=False)
    return df_out, summary_df

df, outlier_report = flag_iqr_outliers(df, CONFIG["numeric_cols"])

            column  lower_bound  upper_bound  outlier_count  outlier_pct
  time_in_hospital         -4.0         12.0           2252         2.21
num_lab_procedures         -8.0         96.0            143         0.14
    num_procedures         -3.0          5.0           4954         4.87
   num_medications         -5.0         35.0           2557         2.51
 number_outpatient          0.0          0.0          16739        16.45
  number_emergency          0.0          0.0          11383        11.19
  number_inpatient         -1.5          2.5           7049         6.93
  number_diagnoses          1.5         13.5            281         0.28

Rows with at least one outlier: 34,482 (33.9%)


##  Outlier Detection and Flagging — Findings

**Purpose:**
Detect statistical outliers in all numeric columns using the IQR method
(multiplier = 1.5). Clinical data outliers are not automatically errors
— a patient with 40 prior emergency visits is unusual but clinically
possible. The rule is flag, never drop. Flag columns travel into
PostgreSQL so Power BI can toggle between views with and without outliers.

**IQR Bounds and Outlier Counts:**

| Column | Lower Bound | Upper Bound | Outlier Count | Outlier % |
|---|---|---|---|---|
| time_in_hospital | -4.0 | 12.0 | 2,252 | 2.21% |
| num_lab_procedures | -8.0 | 96.0 | 143 | 0.14% |
| num_procedures | -3.0 | 5.0 | 4,954 | 4.87% |
| num_medications | -5.0 | 35.0 | 2,557 | 2.51% |
| number_outpatient | 0.0 | 0.0 | 16,739 | 16.45% |
| number_emergency | 0.0 | 0.0 | 11,383 | 11.19% |
| number_inpatient | -1.5 | 2.5 | 7,049 | 6.93% |
| number_diagnoses | 1.5 | 13.5 | 281 | 0.28% |

**Total rows with at least one outlier: 34,482 (33.9%)**

**Key Observations:**

1. **number_outpatient and number_emergency have upper bound of 0.0**
   — this is not an error in the calculation. It means Q3 for both
   columns is 0, confirmed by the numeric profile in Step 5 where
   both columns showed Q1 = Q3 = 0. At least 75% of patients had
   zero outpatient and zero emergency visits in the prior year.
   Any value above 0 is therefore flagged as a statistical outlier
   by the IQR method. This explains why 16,739 (16.45%) and 11,383
   (11.19%) rows are flagged for these two columns respectively.
   These are not errors — they are patients with prior utilization
   which is itself a meaningful clinical signal for readmission risk.

2. **num_procedures has the highest outlier rate at 4.87%** — 4,954
   encounters had more than 5 procedures. Given the range is 0 to 6
   and the upper IQR bound is 5.0, patients with exactly 6 procedures
   are flagged. This is a real clinical scenario not a data error.

3. **num_lab_procedures has the lowest outlier rate at 0.14%** —
   only 143 encounters had more than 96 lab procedures. This is
   consistent with the near-normal distribution observed in Step 5
   (skew: -0.24). The wide IQR of 26 (31 to 57) means most patients
   cluster reasonably around the mean of 43 procedures.

4. **time_in_hospital flags stays longer than 12 days** — 2,252
   encounters (2.21%) had length of stay exceeding 12 days. Given
   the dataset cap of 14 days, these are patients at the upper end
   of the allowed range. Longer stays are clinically associated with
   higher readmission risk — these outliers are analytically important
   and must not be dropped.

5. **number_inpatient lower bound of -1.5** — negative lower bounds
   appear in several columns because the IQR formula can produce
   negative values when Q1 is at or near zero. Since counts cannot
   be negative, the effective lower bound is 0. No actual negative
   values exist in the data — only the upper bound is meaningful
   for these columns.

6. **33.9% of rows have at least one outlier** — this is a high
   proportion but is clinically expected in a diabetes inpatient
   population. Patients with diabetes frequently have complex
   comorbidities that drive high utilization across multiple
   dimensions simultaneously. Dropping 34,482 rows would remove
   a third of the dataset and introduce severe selection bias
   into all readmission rate calculations.

**Flag Columns Added:**
- time_in_hospital_outlier_flag
- num_lab_procedures_outlier_flag
- num_procedures_outlier_flag
- num_medications_outlier_flag
- number_outpatient_outlier_flag
- number_emergency_outlier_flag
- number_inpatient_outlier_flag
- number_diagnoses_outlier_flag
- any_outlier_flag (composite — 1 if outlier in any column)

**Clinical Decision — Flag Not Drop:**
All 34,482 outlier rows are retained in the dataset. In clinical
analytics, unusual values frequently represent the most important
patients — those with the highest complexity and highest readmission
risk. Removing them would make the analysis cleaner statistically
but less representative of the real patient population that drives
hospital readmission rates. The any_outlier_flag column allows
Power BI to offer a toggle between the full population view and
the outlier-excluded view without permanently discarding any data.

**Next Step:** Feature engineering — create five derived clinical
features before export to PostgreSQL.

In [13]:
import sys
import pandas as pd
from config import CONFIG
from sqlalchemy import create_engine, text

engine = create_engine(CONFIG["db_url"])


def export_clean_table(df, engine, schema, table):
    """Truncates existing data and appends new rows without dropping the table structure or views."""
    print("1. Connecting to PostgreSQL and truncating old records...")
    sys.stdout.flush()

    # Truncate inside a transaction block to preserve table structure and views
    with engine.begin() as conn:
        conn.execute(text(f'TRUNCATE TABLE "{schema}"."{table}" RESTART IDENTITY;'))

        print(
            f"2. Exporting {len(df):,} rows to PostgreSQL (please wait for completion)..."
        )
        sys.stdout.flush()

        df.to_sql(
            name=table,
            con=conn,
            schema=schema,
            if_exists="append",  # Appends rows instead of dropping table
            index=False,
            method="multi",
            chunksize=1000,
        )

    # Confirm row count
    count = pd.read_sql(
        f'SELECT COUNT(*) FROM "{schema}"."{table}"', engine
    ).iloc[0, 0]
    print(
        f"\n✅ SUCCESS! Exported {len(df):,} rows  |  PostgreSQL confirms: {count:,} rows"
    )


# Run export
export_clean_table(df, engine, CONFIG["schema"], CONFIG["clean_table"])

1. Connecting to PostgreSQL and truncating old records...
2. Exporting 101,766 rows to PostgreSQL (please wait for completion)...

✅ SUCCESS! Exported 101,766 rows  |  PostgreSQL confirms: 101,766 rows
